<a href="https://colab.research.google.com/github/jhhlim/LLMFundamentals/blob/main/Jason_Lim_hw_2_word2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class 2 — LLM Fundamentals (UCSC Extension)
# Homework 2 Part 2: Product Recommendation with Word2Vec

**Student:** Jason Lim  
**Option chosen:** **2** adapt the *songs recommendation* framework from Class Lab 2a to a **products** dataset  
**Submit to:** svagarwa@ucsc.edu (one Colab notebook link)

Lab 2a treats **songs as tokens** and **playlists as sentences**, then trains Word2Vec so co-occurring songs land near each other in embedding space.  
This notebook keeps that same pipeline and only swaps the domain:

| Songs lab (Lab 2a) | Products (this HW) |
|---|---|
| Song ID = token | `StockCode` = token |
| Playlist = sentence | Customer purchase history = sentence |
| Recommend similar songs | Recommend similar products |

**Dataset (class):** [Online Retail II (UCI) on Kaggle](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)  
**Colab-friendly source used here:** same Online Retail schema via Databricks' public CSV mirror (no Kaggle login required).

In [1]:
%%capture
!pip install -q gensim pandas

## Step 1 Load the products dataset

Homework note from Lab 2a:
> group all products (`StockCodes`) bought by the customer (`CustomerID`)

That is the direct analogue of building playlists from song IDs.

In [2]:
from pathlib import Path
from urllib import request

import pandas as pd

# Public mirror of the classic Online Retail table (same columns as the UCI / Kaggle HW dataset).
DATA_URL = (
    "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/"
    "master/data/retail-data/all/online-retail-dataset.csv"
)
LOCAL_CANDIDATES = [
    Path("online_retail_sample.csv"),  # optional file uploaded next to this notebook
    Path("online_retail.csv"),
]

def load_retail() -> pd.DataFrame:
    for path in LOCAL_CANDIDATES:
        if path.exists():
            print(f"Loading local file: {path}")
            return pd.read_csv(path)

    print("Downloading Online Retail CSV (Colab-friendly, no Kaggle auth)...")
    out = Path("online_retail.csv")
    request.urlretrieve(DATA_URL, out)
    print(f"Saved {out} ({out.stat().st_size / 1e6:.1f} MB)")
    return pd.read_csv(out)

df_raw = load_retail()
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
df_raw.head()

Saved online_retail.csv (45.0 MB)
Shape: (541909, 8)
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## Step 2 — Clean + build customer "playlists" of products

Mirror of Lab 2a:
- drop playlists with only one song → drop customers with only one product
- each remaining list becomes one training sentence for Word2Vec

In [2]:
# --- Cleaning (comments = analysis explanation for the grader) ---
# 1) Keep rows with a known customer and product code.
# 2) Drop cancelled invoices (InvoiceNo starting with 'C') — returns are not purchase co-occurrence.
# 3) Keep positive quantities only.
# 4) Cap to a sample if the full file is huge so Colab stays snappy.

df = df_raw.copy()
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")

df = df.dropna(subset=["CustomerID", "StockCode"])
df = df[~df["InvoiceNo"].str.startswith("C")]
df = df[df["Quantity"] > 0]

# Optional speed knob for Colab — full file still works, this just shortens train time.
MAX_ROWS = 120_000
if len(df) > MAX_ROWS:
    df = df.head(MAX_ROWS).copy()
    print(f"Using first {MAX_ROWS:,} clean rows for faster training.")

df["CustomerID"] = df["CustomerID"].astype(int).astype(str)

print("Clean shape:", df.shape)
print("Unique customers:", df["CustomerID"].nunique())
print("Unique products:", df["StockCode"].nunique())
df.head()

Using first 120,000 clean rows for faster training.
Clean shape: (120000, 8)
Unique customers: 2467
Unique products: 3006


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom


In [3]:
# Group ALL products bought by each customer into one sequence.
# Lab homework wording: "group all products (StockCodes) bought by the customer (CustomerID)"
#
# Analogy to Lab 2a songs:
#   playlists[i] = [song_id, song_id, ...]   ←→   purchases[i] = [stock_code, stock_code, ...]

purchases = (
    df.groupby("CustomerID")["StockCode"]
    .apply(lambda codes: list(dict.fromkeys(codes)))  # preserve order, drop exact dupes in-sequence
    .tolist()
)

# Same filter as songs lab: remove "playlists" with only one item
purchases = [seq for seq in purchases if len(seq) > 1]

print(f"Customer purchase sequences (sentences): {len(purchases):,}")
print("Example customer #1 products:", purchases[0][:15], "...")
print("Example customer #2 products:", purchases[1][:15], "...")

Customer purchase sequences (sentences): 2,390
Example customer #1 products: ['85116', '22375', '71477', '22492', '22771', '22772', '22773', '22774', '22775', '22805', '22725', '22726', '22727', '22728', '22729'] ...
Example customer #2 products: ['84992', '22951', '84991', '21213', '22616', '21981', '21982', '21725', '21211', '84988', '22952', '21977', 'POST', '21980', '21985'] ...


In [4]:
# Product metadata lookup (like songs_df with title/artist in Lab 2a)
products_df = (
    df[["StockCode", "Description"]]
    .dropna()
    .drop_duplicates(subset=["StockCode"])
    .set_index("StockCode")
)
print(products_df.head(10))

                                   Description
StockCode                                     
85123A      WHITE HANGING HEART T-LIGHT HOLDER
71053                      WHITE METAL LANTERN
84406B          CREAM CUPID HEARTS COAT HANGER
84029G     KNITTED UNION FLAG HOT WATER BOTTLE
84029E          RED WOOLLY HOTTIE WHITE HEART.
22752             SET 7 BABUSHKA NESTING BOXES
21730        GLASS STAR FROSTED T-LIGHT HOLDER
22633                   HAND WARMER UNION JACK
22632                HAND WARMER RED POLKA DOT
84879            ASSORTED COLOUR BIRD ORNAMENT


## Step 3 — Train Word2Vec (same API as Lab 2a songs)

Docs: https://radimrehurek.com/gensim/models/word2vec.html

- `vector_size=32` — embedding dimension (same as songs lab)
- `window=10` — how many neighboring products in a customer's history count as context
- `min_count=2` — ignore ultra-rare products (slightly stricter than songs lab's `min_count=1` for retail noise)

In [7]:
import sys
if 'gensim' not in sys.modules:
    !pip install -q gensim

from gensim.models import Word2Vec

# Train embeddings so products that co-occur in customer histories are close in vector space.
model = Word2Vec(
    purchases,
    vector_size=32,  # Dimensionality of product vectors
    window=10,       # Context window within a customer's purchase sequence
    min_count=2,     # Drop products that appear < 2 times across customers
    workers=4,
    sg=0,            # 0=CBOW (default); set sg=1 for skip-gram if you want to experiment
)

print("Vocabulary size (products with embeddings):", len(model.wv))
print("Vector size:", model.wv.vector_size)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 67.7 MB/s eta 0:00:00
Vocabulary size (products with embeddings): 2726
Vector size: 32


## Step 4 — Recommend products (Lab 2a `most_similar` pattern)

In [9]:
import numpy as np

def print_recommendations(stock_code: str, topn: int = 5):
    """Return product descriptions for nearest neighbors of stock_code."""
    stock_code = str(stock_code)
    if stock_code not in model.wv:
        raise KeyError(f"{stock_code} not in vocabulary (try another StockCode)")

    similar_ids = np.array(model.wv.most_similar(positive=[stock_code], topn=topn))[:, 0]
    # Keep only IDs we have descriptions for
    known = [pid for pid in similar_ids if pid in products_df.index]
    return products_df.loc[known]


# Pick a seed product that exists in the trained vocab
seed = next(code for code in products_df.index if code in model.wv)
print("Seed product:")
print(products_df.loc[[seed]])
print("\nTop recommendations:")
print(print_recommendations(seed))

Seed product:
                                  Description
StockCode                                    
85123A     WHITE HANGING HEART T-LIGHT HOLDER

Top recommendations:
                                 Description
StockCode                                   
82494L           WOODEN FRAME ANTIQUE WHITE 
22469                  HEART OF WICKER SMALL
21754               HOME BUILDING BLOCK WORD
21755               LOVE BUILDING BLOCK WORD
82482      WOODEN PICTURE FRAME WHITE FINISH


In [10]:
# Try a couple more seeds from the vocabulary (same idea as Lab 2a testing song 3822 then 3000)
demo_seeds = [code for code in list(model.wv.index_to_key)[:50] if code in products_df.index][:3]

for seed in demo_seeds:
    print("=" * 60)
    print("SEED:", seed, "→", products_df.loc[seed, "Description"])
    print(print_recommendations(seed))
    print()

SEED: 85123A → WHITE HANGING HEART T-LIGHT HOLDER
                                 Description
StockCode                                   
82494L           WOODEN FRAME ANTIQUE WHITE 
22469                  HEART OF WICKER SMALL
21754               HOME BUILDING BLOCK WORD
21755               LOVE BUILDING BLOCK WORD
82482      WOODEN PICTURE FRAME WHITE FINISH

SEED: 22423 → REGENCY CAKESTAND 3 TIER
                                Description
StockCode                                  
22698        PINK REGENCY TEACUP AND SAUCER
22699      ROSES REGENCY TEACUP AND SAUCER 
22776           SWEETHEART CAKESTAND 3 TIER
22697       GREEN REGENCY TEACUP AND SAUCER
22427               ENAMEL FLOWER JUG CREAM

SEED: 22720 → SET OF 3 CAKE TINS PANTRY DESIGN 
                                 Description
StockCode                                   
22722      SET OF 6 SPICE TINS PANTRY DESIGN
22666        RECIPE BOX PANTRY YELLOW DESIGN
22960               JAM MAKING SET WITH JARS
22961        

## Step 5 Quick embedding sanity check

Cosine similarity between two product vectors should be high when they often appear in the same customer baskets.

In [11]:
a, b, c = demo_seeds[0], demo_seeds[1], demo_seeds[2]

print(f"sim({a}, {b}) = {model.wv.similarity(a, b):.4f}")
print(f"sim({a}, {c}) = {model.wv.similarity(a, c):.4f}")
print(f"sim({b}, {c}) = {model.wv.similarity(b, c):.4f}")
print("\nMost similar pairs are not required to be the same category —")
print("Word2Vec only knows co-purchase statistics, not product taxonomy.")

sim(85123A, 22423) = 0.8822
sim(85123A, 22720) = 0.7016
sim(22423, 22720) = 0.9214

Most similar pairs are not required to be the same category —
Word2Vec only knows co-purchase statistics, not product taxonomy.


## Learnings / analysis notes

- **Songs → products mapping:** song ID ↔ `StockCode`; playlist ↔ customer purchase list; `most_similar` ↔ product recommendations.
- **Why group by `CustomerID`?** The HW asks for products a customer bought together over time. That creates longer "sentences" than single invoices and matches the playlist analogy.
- **What the embedding captures:** not product text meaning (we never embed the Description string) — only **basket co-occurrence**.
- **Failure modes:** rare products (`min_count`), cold-start new SKUs, and cancelled/return noise if not filtered.
- **Class 2 link:** same embedding idea as Lab 2a GloVe demos — vectors + cosine similarity — but here we *train* the vectors from retail sequences instead of loading pretrained word vectors.

---

**References**
- Class Lab 2a — *Recommending songs by embeddings*
- Homework dataset: https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci
- Gensim Word2Vec: https://radimrehurek.com/gensim/models/word2vec.html